<a href="https://colab.research.google.com/github/antonyrichard29/LLM-finetuning/blob/main/HealthMate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
torch.cuda.empty_cache()

In [3]:
!pip install transformers datasets accelerate peft bitsandbytes trl fsspec==2025.7.0

INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.6/199.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 12.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully u

In [4]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

In [5]:
# Load model & tokenizer
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, torch_dtype=torch.float16, device_map="auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
data = [
    {
        "instruction": "I have a sore throat.",
        "response": "Gargle with warm salt water, drink warm fluids like tea with honey, and avoid cold drinks. If it lasts more than a few days, consult a doctor."
    },
    {
        "instruction": "I have a headache.",
        "response": "Rest in a quiet, dark room, stay hydrated, and take a mild pain reliever like paracetamol if needed. Avoid stress and bright screens."
    },
    {
        "instruction": "I have a cold.",
        "response": "Drink warm fluids, rest well, use steam inhalation, and take vitamin C. Avoid cold weather exposure."
    },
    {
        "instruction": "I have a fever.",
        "response": "Stay hydrated, rest, and take paracetamol if temperature is high. Use a cool damp cloth on your forehead to reduce heat."
    },
    {
        "instruction": "I feel dizzy.",
        "response": "Sit or lie down immediately, drink water, and avoid sudden movements. Eat something light if you haven’t eaten in a while."
    },
    {
        "instruction": "I have a cough.",
        "response": "Drink warm water with honey, use steam inhalation, and avoid cold or oily foods. If cough persists more than a week, see a doctor."
    },
    {
        "instruction": "I have stomach pain.",
        "response": "Drink warm water, eat light foods, avoid spicy and oily meals, and rest. If pain is severe or continues, consult a doctor."
    },
    {
        "instruction": "I feel bloated.",
        "response": "Avoid carbonated drinks, eat smaller meals, reduce salt intake, and go for a light walk after eating."
    },
    {
        "instruction": "I have a backache.",
        "response": "Apply a warm compress, maintain proper posture, do gentle stretching, and avoid lifting heavy objects."
    },
    {
        "instruction": "I have a toothache.",
        "response": "Rinse your mouth with warm salt water, apply a cold compress outside the cheek, and avoid sweet foods until you see a dentist."
    },
    {
        "instruction": "I feel tired all the time.",
        "response": "Get enough sleep, stay hydrated, eat a balanced diet with fruits and vegetables, and avoid excessive caffeine."
    },
    {
        "instruction": "I have an earache.",
        "response": "Apply a warm compress to the ear, avoid poking inside, and keep the ear dry. If pain persists, visit a doctor."
    },
    {
        "instruction": "I have diarrhea.",
        "response": "Drink plenty of oral rehydration solution (ORS), eat plain rice or bananas, and avoid oily or spicy food."
    },
    {
        "instruction": "I am constipated.",
        "response": "Eat high-fiber foods like fruits and vegetables, drink more water, and do light exercise daily."
    },
    {
        "instruction": "I have muscle cramps.",
        "response": "Stretch the affected muscle, massage gently, apply a warm compress, and stay hydrated with electrolyte-rich drinks."
    }
]

# Prepare dataset: format Q&A into instruction-response text
dataset = Dataset.from_list(data)

def format_prompt(example):
    return {
        "text": f"### Instruction:\nQ: {example['instruction']}\n\n### Response:\nA: {example['response']}"
    }

dataset = dataset.map(format_prompt)
dataset = dataset.remove_columns([col for col in dataset.column_names if col != "text"])

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

In [7]:
model = prepare_model_for_kbit_training(model)

# Configure and apply LoRA for causal language modeling
lora_config = LoraConfig(
    r=8, # Rank of low-rank matrices (smaller r → fewer trainable params, faster training)
    lora_alpha=16, # Scaling factor to control the impact of LoRA updates
    target_modules=["q_proj","v_proj","k_proj","o_proj"], # Specific transformer layers where LoRA is applied
    lora_dropout=0.05,  # Dropout applied to LoRA layers for regularization
    bias="none",  # Whether to train bias terms ("none", "all", or "lora_only")
    task_type=TaskType.CAUSAL_LM # Task type, here set for Causal Language Modeling
)
model = get_peft_model(model, lora_config)

In [8]:
# Prepares tokenized dataset for training by applying the tokenizer with truncation and padding
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

tokenized_dataset = dataset.map(
    lambda x: tokenizer(x['text'], truncation=True, padding="max_length", max_length=256),
    batched=True
)

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

In [9]:
args = TrainingArguments(
    output_dir="./tinyllama-medbot",     # Folder where trained model & checkpoints will be saved
    num_train_epochs=3,                  # Train for 3 full passes over the dataset
    per_device_train_batch_size=2,       # Each GPU/CPU processes 2 samples per step
    gradient_accumulation_steps=4,       # Accumulate gradients for 4 steps before updating weights (effective batch = 2×4=8)
    learning_rate=2e-4,                  # How fast the model learns (0.0002)
    fp16=True,                          # Use half-precision (faster, less memory, if GPU supports it)
    bf16 = False,                         # Use bfloat16 (faster, less memory, if TPU supports it)
    optim="adafactor",                   # AdamW causes fused errors on TPU
    ddp_find_unused_parameters=False,    # Reduce TPU sync overhead
    logging_steps=10,                    # Log training progress every 10 steps
    save_steps=100,                      # Save a checkpoint every 100 steps
    save_total_limit=2,                  # Keep only the latest 2 checkpoints (older ones deleted)
    report_to="none"                     # Disable logging to tools like WandB or TensorBoard

)

trainer = Trainer(
    model=model,   # The model being fine-tuned (LoRA-applied TinyLlama here)
    args=args,     # The training arguments defined above
    train_dataset=tokenized_dataset,   # Training data after tokenization
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss


TrainOutput(global_step=6, training_loss=2.1297451655069985, metrics={'train_runtime': 17.6572, 'train_samples_per_second': 2.549, 'train_steps_per_second': 0.34, 'total_flos': 71661209518080.0, 'train_loss': 2.1297451655069985, 'epoch': 3.0})

In [10]:
# Move model to CPU before saving
model.to("cpu")

model.save_pretrained("healthmate_finetuned")
tokenizer.save_pretrained("healthmate_finetuned")

('healthmate_finetuned/tokenizer_config.json',
 'healthmate_finetuned/special_tokens_map.json',
 'healthmate_finetuned/chat_template.jinja',
 'healthmate_finetuned/tokenizer.model',
 'healthmate_finetuned/added_tokens.json',
 'healthmate_finetuned/tokenizer.json')

In [11]:
# Compress the 'healthmate_finetuned' folder into 'healthmate_finetuned.zip' (recursive, includes all files/subfolders)
!zip -r healthmate_finetuned.zip healthmate_finetuned

  adding: healthmate_finetuned/ (stored 0%)
  adding: healthmate_finetuned/adapter_model.safetensors (deflated 8%)
  adding: healthmate_finetuned/tokenizer.model (deflated 55%)
  adding: healthmate_finetuned/tokenizer.json (deflated 85%)
  adding: healthmate_finetuned/adapter_config.json (deflated 58%)
  adding: healthmate_finetuned/special_tokens_map.json (deflated 79%)
  adding: healthmate_finetuned/chat_template.jinja (deflated 60%)
  adding: healthmate_finetuned/tokenizer_config.json (deflated 69%)
  adding: healthmate_finetuned/README.md (deflated 66%)


In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from peft import PeftModel

model_path = "/content/healthmate_finetuned"
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Load the base model first
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
base_model = AutoModelForCausalLM.from_pretrained(base_model_id, trust_remote_code=True, torch_dtype=torch.float16, device_map="auto")

# Load the PEFT model on top of the base model
model = PeftModel.from_pretrained(base_model, model_path)

model.eval()

def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():              # ensures no gradients (faster inference)
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,        # how long the reply can be
            do_sample=True,            # enables sampling (not greedy)
            temperature=0.7,           # creativity (lower = more focused, higher = more random)
            top_p=0.9,                 # nucleus sampling (limits to top 90% probability mass)
            repetition_penalty=1.2,    # avoids repeating same words
            eos_token_id=tokenizer.eos_token_id   # stops generation at EOS
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [13]:
while True:
    user_input = input("Enter your symptoms (or type 'exit' to quit): ")

    if user_input.lower() == "exit":
        break

    prompt = f"### Instruction:\nQ: {user_input}\n\n### Response:\n"
    response = generate_response(prompt)

    # Extract only after "### Response:" to keep output clean
    clean_response = response.split("### Response:")[-1].strip()

    print("\nDiagnosis Suggestion:", clean_response, "\n")

Enter your symptoms (or type 'exit' to quit): cough,running nose

Diagnosis Suggestion: A: take a cold medicine or sneeze into your hand and throw it away. Be sure to wash your hands afterwards! 

Enter your symptoms (or type 'exit' to quit): fever

Diagnosis Suggestion: A: Fever is a symptom of the following diseases. Please select the disease for which you have been diagnosed with fever. 1) Infectious Disease (e.g., Typhoid, Hepatitis A, Cholera, Lyme Disease) 2) Endocrine Disorder (e.g., Hypothyroidism, Diabetes Mellitus, Cushing Syndrome, Pituitary Tumor) 3) Metabolic Disorder (e.g., Hyperthyroidism, Congenital Adrenal Hyperplasia, Gaucher's Disease) 4) Cardiovascular Disorder (e.g., Coronary Artery Disease, Heart Failure, Angina Pectoris) 5) Neurological Disorders (e.g., Alzheimer’s Disease, Parkinson’s Disease, Huntington’s Disease) 6) Respiratory Disorder (e.g., Asthma, Chronic Obstructive Pulmonary Disease, Emphysema) 7) Genetic Disorder (e.g., Thyrogenesis, Hemophilia B) 8) Sk